# Generate Drug Response Datasets

This code is used for generating drugs reponse datasets specific to a set of drugs

In [1]:
import pandas as pd
import numpy as np
import pubchempy as pcp
from sklearn.model_selection import train_test_split
from sklearn.model_selection import LeavePOut, KFold

In [2]:
whole_data = pd.read_csv("whole_dataset_2.txt", sep = '\t', header = None, names = ["cell-line", "drug", "response", "source"])
cellline_df = pd.read_csv("cell2ind_av.txt", sep='\t', header=None)
celllines = cellline_df[1].values
whole_data = whole_data[(whole_data["cell-line"].isin(celllines))]
len(whole_data)

450382

In [6]:
train_data = pd.read_csv("train_dataset_rsi.txt", sep = '\t', header = None, names = ["cell-line", "drug", "response", "source"])
len(train_data)

10472

In [7]:
len(list(set(list(train_data["cell-line"]))))

1132

In [8]:
len(list(set(list(whole_data["drug"]))))

636

## Select the Drugs needed in a dataset

In [13]:
# Use this to change the drugs I'm interested in for building any datasets
rsi_drugs = ['Etoposide', 'Camptothecin', 'Gemcitabine','CD437', 'Olaparib', 'MK-1775', '5-FU', 
             'Cisplatin', 'Ceralasertib', 'Bleomycin-a2', 'Doxorubicin', 'Methotrexate', 'Mitomycin-C', 
             'Nelarabine', 'Irinotecan', 'Oxaliplatin', 'Epirubicin', 'Topotecan', 'Teniposide', 
             'Mitoxantrone', 'Fludarabine', 'Veliparib', 'Niraparib', 'Talazoparib', 'Pyridostatin']
non_rsi_drugs = ["Tamoxifen", "Sorafenib", "Dasatinib", "Saracatinib", "Erlotinib", "Palbociclib", "Paclitaxel", 
                 "Bortezomib", "Manumycin-A", "Fedratinib", "Daporinad", "Navitoclax", "Erastin", "Rapamycin",
                 "Imatinib", "Docetaxel", "Vincristine", "Vinblastine", "Vinorelbine", "Gefitinib", "Afatinib", 
                 "Osimertinib", "Dinaciclib", "Ribociclib", "Lapatinib"]

drug_names = rsi_drugs + non_rsi_drugs
print("Total drugs to check:", len(drug_names))
drug_smiles = []
smiles_to_name = {}
for drug in drug_names:
    cid = pcp.get_cids(drug, 'name')
    if len(cid) > 0:
        c = pcp.Compound.from_cid(cid[0])
        drug_smiles.append(c.isomeric_smiles)
        smiles_to_name[c.isomeric_smiles]= drug
drug_smiles.append('N.N.[Cl-].[Cl-].[Pt+2]')
smiles_to_name['N.N.[Cl-].[Cl-].[Pt+2]'] = 'Cisplatin'

drug_data = whole_data[(whole_data["drug"].isin(drug_smiles))]
drug_data = drug_data[(drug_data["cell-line"].isin(celllines))]
kept_drugs = drug_data.drop_duplicates(subset = ["drug"])
kept_drugs = list(kept_drugs['drug'])
kept_drugs = [smiles_to_name[smiles] for smiles in kept_drugs]
print("Data size:", len(drug_data), '\n')
print("Drugs used in dataset:", len(kept_drugs))
for drug in kept_drugs:
    print(drug)
print('\n')
print("Drugs not in datasets:", len(set(drug_names).difference(set(kept_drugs))))
for drug in set(drug_names).difference(set(kept_drugs)):
    print(drug)

Total drugs to check: 50
Data size: 42563 

Drugs used in dataset: 50
Camptothecin
Afatinib
Dinaciclib
Topotecan
Teniposide
Mitoxantrone
Fludarabine
Nelarabine
Vincristine
Docetaxel
5-FU
Irinotecan
Oxaliplatin
Niraparib
Sorafenib
Vinblastine
Cisplatin
Erlotinib
Gefitinib
MK-1775
Gemcitabine
Bortezomib
Tamoxifen
Talazoparib
Ceralasertib
Olaparib
Palbociclib
Osimertinib
Epirubicin
Lapatinib
Dasatinib
Paclitaxel
Navitoclax
Pyridostatin
Vinorelbine
Rapamycin
Ribociclib
Daporinad
Methotrexate
Doxorubicin
Etoposide
Manumycin-A
Mitomycin-C
Erastin
Imatinib
CD437
Veliparib
Saracatinib
Bleomycin-a2
Fedratinib


Drugs not in datasets: 0


## For Interpretability ... Save the Whole Dataset

In [3]:
dataset_name = "all_drug_response.txt"
whole_data.to_csv(dataset_name, columns=None, index=False, sep='\t', header=None)

## For Training ... Save the data as 5-fold train/val/test

In [15]:
drug_type = "half_half" # Change this depending on what drugs I'm generating data for
celllines = np.asarray(list(set(drug_data["cell-line"])), dtype='object')
kf = KFold(n_splits=5, shuffle=True)
for i, (train_index, test_index) in enumerate(kf.split(celllines)):
    train_celllines = celllines[train_index]
    test_celllines = celllines[test_index]
    train_celllines, val_celllines = train_test_split(train_celllines, test_size=0.05)
    train_dataset = drug_data.loc[drug_data['cell-line'].map(lambda a: a in train_celllines)]
    val_dataset = drug_data.loc[drug_data['cell-line'].map(lambda a: a in val_celllines)]
    test_dataset = drug_data.loc[drug_data['cell-line'].map(lambda a: a in test_celllines)]
    train_dataset.columns = ["cellline", "drug", "response", "source"]
    train_dataset.to_csv("train_dataset_"+drug_type+"_%d.txt"%i, columns=None, index=False, sep='\t', header=None)
    val_dataset.columns = ["cellline", "drug", "response", "source"]
    val_dataset.to_csv("val_dataset_"+drug_type+"_%d.txt"%i, columns=None, index=False, sep='\t', header=None)
    test_dataset.columns = ["cellline", "drug", "response", "source"]
    test_dataset.to_csv("test_dataset_"+drug_type+"_%d.txt"%i, columns=None, index=False, sep='\t', header=None)